In [8]:
from Bio import SeqIO
import os
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
from Bio import SeqIO
import os
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, AutoTokenizer,BertModel, AutoModel 
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.nn.functional as F

In [9]:
def getFastaData(fasta_path):
    fasta_file = fasta_path
    sequences = []
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequences.append({
            "id": record.id,
            "sequence": str(record.seq)
        })

    result = []
    for sequence in sequences:
        result.append(sequence["sequence"])
    return result

In [10]:
p = getFastaData(r'./buchong/p_test09.fasta')
n = getFastaData(r'./buchong/n_test09.fasta')
p_label = [1 for i in range(len(p))]
n_label = [0 for i in range(len(n))]
val_features_first = p + n
val_labels_first = p_label + n_label

In [11]:
class myModel(torch.nn.Module):
    def __init__(self,esm2):
        super(myModel,self).__init__()
        self.esm2 = esm2
        self.fc1 = torch.nn.Linear(632,32)
        self.fc2 = torch.nn.Linear(32,8)
        self.fc3 = torch.nn.Linear(8,1)
        
        self.conv1 = nn.Conv1d(1, 8, kernel_size=3)
        self.conv2 = nn.Conv1d(8, 2, kernel_size=3)
        self.sigmoid = nn.Sigmoid()
        self.relu = nn.ReLU()  
        self.dropout1 = nn.Dropout(p=0.1)
        self.dropout2 = nn.Dropout(p=0.3)
    def forward(self,x):
        outputs_ = self.esm2(**inputs)
        x = outputs_.last_hidden_state[:, 0, :]
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout1(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

In [12]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, 
    matthews_corrcoef, recall_score, precision_score, 
    confusion_matrix, average_precision_score
)
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

def comp_result(y_test, y_pred, y_proba):
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    mcc = matthews_corrcoef(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    auprc = average_precision_score(y_test, y_proba)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC: {auc:.4f}")
    print(f"MCC: {mcc:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Specificity (SP): {specificity:.4f}")
    print(f"AUPRC: {auprc:.4f}")
    return accuracy, f1, auc, mcc, recall, precision, specificity, auprc

In [ ]:
from sklearn.model_selection import KFold
import numpy as np
import copy as cp
import pandas as pd
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cache_directory = r"/home/hqzhang/neuropeptide prediction/model"
tokenizer_ = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D", cache_dir=cache_directory)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
for kf_in in range(1,6):
    print("================%d================"%kf_in)
    model = torch.load(r'./model result/self_20260711model_09_%d.pth' % kf_in)
    model.to(device)
    model.eval()
    predictions_list = []
    outputs_list = []
    for i in range(len(val_features_first)):
        inputs = tokenizer_([val_features_first[i]], return_tensors='pt', padding=True, truncation=True)
        inputs = inputs.to(device)
        outputs = model(inputs)
        outputs = outputs.view(-1)
        predictions = (outputs > 0.5).float()
        outputs_list.append(outputs.tolist()[0])
        predictions_list.append(predictions.tolist()[0])
    comp_result(val_labels_first, predictions_list, outputs_list)
    val_labels_first_pd = pd.DataFrame(val_labels_first)
    outputs_pd = pd.DataFrame(outputs_list)
    val_labels_first_pd.to_csv('./result/self_09_Fold%d_label.csv' % kf_in)
    outputs_pd.to_csv('./result/self_09_Fold%d_output.csv' % kf_in)